
# RPC pseudo-tracklet efficiency analysis with PyROOT

This notebook upgrades the earlier toy implementation into a **ROOT/PyROOT workflow** consistent with the `rpc-hv-efficiency` package structure:

- event trees named `events`
- vector branches `Strip_X`, `Strip_Y`, `Strip_Tdc_diff`
- scalar branch `nCluster`
- multi-chamber modes:
  - **2 chambers**: `A → B`
  - **3 chambers**: `A + B → C`
  - **4 chambers**: `A + B + C → D`

The notebook implements:

1. **One-HV pseudo-tracklet reconstruction**
2. **Per-HV JSON summaries**
3. **HV scan automation**
4. **CMS-style efficiency plots**

> Adjust paths, chamber ordering, `z` positions, and ROI definitions to match your setup.


In [ ]:

from __future__ import annotations

import json
import math
import subprocess
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ROOT

ROOT.gROOT.SetBatch(True)
print("ROOT version:", ROOT.gROOT.GetVersion())


## 1. Physics constants, HV map, and helper utilities

In [ ]:

HV_MAP = {
    "HV1": 6.0, "HV2": 6.5, "HV3": 6.7, "HV4": 6.8, "HV5": 6.9,
    "HV6": 7.0, "HV7": 7.1, "HV8": 7.2, "HV9": 7.3, "HV10": 7.4, "HV11": 7.5,
}

def pick_hit(ev, hit_mode="earliest"):
    xs = np.asarray(ev.Strip_X, dtype=float)
    ys = np.asarray(ev.Strip_Y, dtype=float)
    ts = np.asarray(ev.Strip_Tdc_diff, dtype=float)
    if xs.size == 0 or ys.size == 0 or ts.size == 0:
        return None
    n = min(xs.size, ys.size, ts.size)
    xs, ys, ts = xs[:n], ys[:n], ts[:n]
    if hit_mode == "earliest":
        i = int(np.argmin(ts))
        return float(xs[i]), float(ys[i]), float(ts[i])
    if hit_mode == "centroid":
        return float(xs.mean()), float(ys.mean()), float(ts.mean())
    raise ValueError(f"Unsupported hit_mode={hit_mode}")

def fit_line(zs, vs):
    z = np.asarray(zs, dtype=float)
    v = np.asarray(vs, dtype=float)
    A = np.vstack([z, np.ones_like(z)]).T
    (a, b), *_ = np.linalg.lstsq(A, v, rcond=None)
    return float(a), float(b)

def compute_eff_hist(h_num, h_den, name="h_eff"):
    h_eff = h_num.Clone(name)
    h_eff.Reset()
    h_eff.Divide(h_num, h_den, 1.0, 1.0, "B")
    return h_eff

def binomial_err(matched, eligible):
    if eligible <= 0:
        return 0.0
    e = matched / eligible
    e = min(max(e, 0.0), 1.0)
    return math.sqrt(e * (1.0 - e) / eligible)

def roi_bin_range(h, x_min, x_max, y_min, y_max):
    ax = h.GetXaxis(); ay = h.GetYaxis()
    bx1 = max(1, min(ax.FindBin(x_min), ax.GetNbins()))
    bx2 = max(1, min(ax.FindBin(x_max), ax.GetNbins()))
    by1 = max(1, min(ay.FindBin(y_min), ay.GetNbins()))
    by2 = max(1, min(ay.FindBin(y_max), ay.GetNbins()))
    if bx2 < bx1: bx1, bx2 = bx2, bx1
    if by2 < by1: by1, by2 = by2, by1
    return bx1, bx2, by1, by2

def summarize_root_map(root_path, hv_kv, roi_rect=None, roi_name="scint_overlap",
                       h_total_name="h_total", h_tracklet_name="h_tracklet", out_json=None):
    f = ROOT.TFile.Open(str(root_path))
    if not f or f.IsZombie():
        raise RuntimeError(f"Cannot open ROOT file: {root_path}")
    h_total = f.Get(h_total_name); h_tracklet = f.Get(h_tracklet_name)
    if not h_total or not h_tracklet:
        raise RuntimeError("Missing h_total and/or h_tracklet in ROOT file")
    h_total.SetDirectory(0); h_tracklet.SetDirectory(0); f.Close()

    if roi_rect is None:
        x_min, x_max = h_total.GetXaxis().GetXmin(), h_total.GetXaxis().GetXmax()
        y_min, y_max = h_total.GetYaxis().GetXmin(), h_total.GetYaxis().GetXmax()
    else:
        x_min, x_max, y_min, y_max = map(float, roi_rect)

    bx1, bx2, by1, by2 = roi_bin_range(h_total, x_min, x_max, y_min, y_max)
    eligible_roi = float(h_total.Integral(bx1, bx2, by1, by2))
    matched_roi  = float(h_tracklet.Integral(bx1, bx2, by1, by2))
    eligible_all = float(h_total.Integral())
    matched_all  = float(h_tracklet.Integral())

    payload = {
        "hv_kv": float(hv_kv),
        "source_root": str(Path(root_path).resolve()),
        "roi": {
            "name": roi_name, "type": "rect",
            "x_min": float(x_min), "x_max": float(x_max),
            "y_min": float(y_min), "y_max": float(y_max),
            "bin_x1": int(bx1), "bin_x2": int(bx2),
            "bin_y1": int(by1), "bin_y2": int(by2),
        },
        "roi_counts": {"eligible": eligible_roi, "matched": matched_roi},
        "roi_efficiency": {
            "eff": (matched_roi / eligible_roi) if eligible_roi > 0 else 0.0,
            "err_binom": binomial_err(matched_roi, eligible_roi),
        },
        "global_counts": {"eligible": eligible_all, "matched": matched_all},
        "global_efficiency": {
            "eff": (matched_all / eligible_all) if eligible_all > 0 else 0.0,
            "err_binom": binomial_err(matched_all, eligible_all),
        },
    }
    if out_json is not None:
        Path(out_json).write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return payload


## 2. Core pseudo-tracklet producer for a single HV point

In [ ]:

def run_tracklet_efficiency_single_hv(
    files, z_positions, out_root, tree_name="events", tag="HVX",
    target_index=-1, anchor_index=0, hit_mode="earliest",
    tol_x=7.0, tol_y=20.0, tol_t=3.0,
    cluster_size_max=6, require_ncluster_eq1=False,
    cluster_size_on_all_ref=False, theta_max_deg=None,
    nbins_x=64, nbins_y=128, x_range=(0.0, 60.0), y_range=(0.0, 140.0),
):
    files = [Path(f) for f in files]
    z_positions = list(map(float, z_positions))
    if len(files) != len(z_positions):
        raise ValueError("files and z_positions must have same length")
    if len(files) < 2:
        raise ValueError("need at least 2 chambers")

    n_ch = len(files)
    if target_index < 0: target_index += n_ch
    if anchor_index < 0: anchor_index += n_ch
    if target_index == anchor_index:
        raise ValueError("target_index cannot equal anchor_index")
    ref_idx = [i for i in range(n_ch) if i != target_index]

    roots, trees = [], []
    for f in files:
        rf = ROOT.TFile.Open(str(f))
        if not rf or rf.IsZombie():
            raise RuntimeError(f"Cannot open {f}")
        tr = rf.Get(tree_name)
        if not tr:
            raise RuntimeError(f"Tree '{tree_name}' not found in {f}")
        roots.append(rf); trees.append(tr)

    n_entries = min(int(t.GetEntries()) for t in trees)
    h_total = ROOT.TH2D("h_total", "Eligible; x_{anchor} [cm]; y_{anchor} [cm]",
                        nbins_x, x_range[0], x_range[1], nbins_y, y_range[0], y_range[1])
    h_tracklet = ROOT.TH2D("h_tracklet", "Matched; x_{anchor} [cm]; y_{anchor} [cm]",
                           nbins_x, x_range[0], x_range[1], nbins_y, y_range[0], y_range[1])

    eligible = matched = ghosts = 0
    for i in range(n_entries):
        for t in trees: t.GetEntry(i)
        ev_anchor = trees[anchor_index]
        ev_target = trees[target_index]

        if int(ev_anchor.nCluster) != 1: continue
        if int(ev_target.nCluster) > 1: continue
        if len(ev_anchor.Strip_Tdc_diff) > cluster_size_max: continue

        hits_ref = {}
        ok = True
        for j in ref_idx:
            ev = trees[j]
            if j != anchor_index:
                if require_ncluster_eq1 and int(ev.nCluster) != 1:
                    ok = False; break
                if cluster_size_on_all_ref and len(ev.Strip_Tdc_diff) > cluster_size_max:
                    ok = False; break
            hit = pick_hit(ev, hit_mode=hit_mode)
            if hit is None:
                ok = False; break
            hits_ref[j] = hit
        if not ok: continue

        eligible += 1
        xA, yA, tA = hits_ref[anchor_index]
        h_total.Fill(xA, yA)

        if int(ev_target.nCluster) == 0:
            ghosts += 1
            continue

        z_ref = [z_positions[j] for j in ref_idx]
        x_ref = [hits_ref[j][0] for j in ref_idx]
        y_ref = [hits_ref[j][1] for j in ref_idx]
        z_t = z_positions[target_index]

        if len(z_ref) >= 2:
            ax, bx = fit_line(z_ref, x_ref)
            ay, by = fit_line(z_ref, y_ref)
            x_pred = ax * z_t + bx
            y_pred = ay * z_t + by
            theta_deg = math.degrees(math.atan2(math.sqrt(ax * ax + ay * ay), 1.0))
            if theta_max_deg is not None and theta_deg > theta_max_deg:
                continue
        else:
            x_pred, y_pred = xA, yA

        t_pred = tA
        hitT = pick_hit(ev_target, hit_mode=hit_mode)
        if hitT is None:
            ghosts += 1
            continue
        xT, yT, tT = hitT

        dx = xT - x_pred; dy = yT - y_pred; dt = tT - t_pred
        if abs(dx) < tol_x and abs(dy) < tol_y and abs(dt) < tol_t:
            matched += 1
            h_tracklet.Fill(xA, yA)

    h_eff = compute_eff_hist(h_tracklet, h_total, "h_eff")
    out_root = Path(out_root)
    out_root.parent.mkdir(parents=True, exist_ok=True)
    fout = ROOT.TFile(str(out_root), "RECREATE")
    h_total.Write(); h_tracklet.Write(); h_eff.Write(); fout.Close()
    for rf in roots: rf.Close()

    return {
        "tag": tag, "n_entries": int(n_entries),
        "eligible": int(eligible), "matched": int(matched), "ghosts": int(ghosts),
        "efficiency": (matched / eligible) if eligible > 0 else 0.0,
        "efficiency_err": binomial_err(matched, eligible),
        "out_root": str(out_root),
    }


## 3. User configuration

In [ ]:

CHAMBER_BASES = [
    Path("/path/to/ChamberA"),
    Path("/path/to/ChamberB"),
    Path("/path/to/ChamberC"),
]

Z_POSITIONS = [0.0, 55.0, 110.0]
ANCHOR_INDEX = 0
TARGET_INDEX = 2
TREE_NAME = "events"
HIT_MODE = "earliest"
TOL_X = 7.0
TOL_Y = 20.0
TOL_T = 3.0
THETA_MAX_DEG = None
ROI_NAME = "scint_overlap"
ROI_RECT = (0.0, 22.0, 0.0, 130.0)

BASE_OUT = Path("out_hv_notebook")
MAP_DIR = BASE_OUT / "maps"
SUM_DIR = BASE_OUT / "summaries"
PLOT_DIR = BASE_OUT / "plots"
for d in [MAP_DIR, SUM_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)


## 4. Run one HV point

In [ ]:

HV_TAG = "HV6"
HV_VALUE = HV_MAP[HV_TAG]
files = [base / HV_TAG / "data.root" for base in CHAMBER_BASES]
print(files)

# Uncomment when paths are ready:
# result = run_tracklet_efficiency_single_hv(
#     files=files,
#     z_positions=Z_POSITIONS,
#     out_root=MAP_DIR / f"tracklets_efficiency_{HV_TAG}.root",
#     tree_name=TREE_NAME,
#     tag=HV_TAG,
#     target_index=TARGET_INDEX,
#     anchor_index=ANCHOR_INDEX,
#     hit_mode=HIT_MODE,
#     tol_x=TOL_X,
#     tol_y=TOL_Y,
#     tol_t=TOL_T,
#     theta_max_deg=THETA_MAX_DEG,
# )
# result


## 5. Summarize one HV point into JSON

In [ ]:

# Example:
# payload = summarize_root_map(
#     root_path=MAP_DIR / f"tracklets_efficiency_{HV_TAG}.root",
#     hv_kv=HV_VALUE,
#     roi_rect=ROI_RECT,
#     roi_name=ROI_NAME,
#     out_json=SUM_DIR / f"summary_{HV_TAG}.json",
# )
# payload


## 6. Loop over all HV directories

In [ ]:

def discover_hv_tags(chamber_bases):
    tags = set()
    for d in chamber_bases[0].glob("HV*"):
        if d.is_dir() and d.name in HV_MAP:
            tags.add(d.name)
    return sorted(tags, key=lambda x: HV_MAP[x])

def run_full_hv_scan(chamber_bases, z_positions, anchor_index, target_index, out_base,
                     tree_name="events", hit_mode="earliest", tol_x=7.0, tol_y=20.0, tol_t=3.0,
                     theta_max_deg=None, roi_name="scint_overlap", roi_rect=(0.0, 22.0, 0.0, 130.0)):
    out_base = Path(out_base)
    map_dir = out_base / "maps"; sum_dir = out_base / "summaries"
    map_dir.mkdir(parents=True, exist_ok=True); sum_dir.mkdir(parents=True, exist_ok=True)
    scan_rows = []

    for hv_tag in discover_hv_tags(chamber_bases):
        files = [Path(base) / hv_tag / "data.root" for base in chamber_bases]
        if not all(f.exists() for f in files):
            print("Skipping", hv_tag, "because one or more files are missing")
            continue
        hv_kv = HV_MAP[hv_tag]
        out_root = map_dir / f"tracklets_efficiency_{hv_tag}.root"
        out_json = sum_dir / f"summary_{hv_tag}.json"
        print(f"Processing {hv_tag} ({hv_kv:.2f} kV)")

        result = run_tracklet_efficiency_single_hv(
            files=files, z_positions=z_positions, out_root=out_root, tree_name=tree_name,
            tag=hv_tag, target_index=target_index, anchor_index=anchor_index,
            hit_mode=hit_mode, tol_x=tol_x, tol_y=tol_y, tol_t=tol_t, theta_max_deg=theta_max_deg,
        )
        summary = summarize_root_map(out_root, hv_kv=hv_kv, roi_rect=roi_rect,
                                     roi_name=roi_name, out_json=out_json)

        scan_rows.append({
            "hv_tag": hv_tag, "hv_kv": hv_kv,
            "eligible": result["eligible"], "matched": result["matched"], "ghosts": result["ghosts"],
            "global_eff": summary["global_efficiency"]["eff"],
            "global_err": summary["global_efficiency"]["err_binom"],
            "roi_eff": summary["roi_efficiency"]["eff"],
            "roi_err": summary["roi_efficiency"]["err_binom"],
        })
    return scan_rows


In [ ]:

# Uncomment to run the entire scan:
# scan_rows = run_full_hv_scan(
#     chamber_bases=CHAMBER_BASES,
#     z_positions=Z_POSITIONS,
#     anchor_index=ANCHOR_INDEX,
#     target_index=TARGET_INDEX,
#     out_base=BASE_OUT,
#     tree_name=TREE_NAME,
#     hit_mode=HIT_MODE,
#     tol_x=TOL_X,
#     tol_y=TOL_Y,
#     tol_t=TOL_T,
#     theta_max_deg=THETA_MAX_DEG,
#     roi_name=ROI_NAME,
#     roi_rect=ROI_RECT,
# )
# scan_rows[:3]


## 7. CMS-style plotting from summary JSON files

In [ ]:

def sigmoid(hv, eps0, eps_max, hv50, k):
    return eps0 + (eps_max - eps0) / (1.0 + np.exp(-(hv - hv50) / k))

def fit_sigmoid(hv, eff):
    hv = np.asarray(hv, float); eff = np.asarray(eff, float)
    eps0_guess = max(0.0, float(np.min(eff)))
    eps_max_guess = min(1.0, float(np.max(eff)))
    hv50_guess = float(hv[np.argmin(np.abs(eff - 0.5 * (eps0_guess + eps_max_guess)))])
    k_guess = 0.05
    try:
        from scipy.optimize import curve_fit
        bounds = ([0.0, 0.0, hv.min() - 1.0, 1e-3], [1.0, 1.0, hv.max() + 1.0, 1.0])
        popt, _ = curve_fit(sigmoid, hv, eff, p0=[eps0_guess, eps_max_guess, hv50_guess, k_guess],
                            bounds=bounds, maxfev=20000)
        return tuple(map(float, popt))
    except Exception:
        return (eps0_guess, eps_max_guess, hv50_guess, k_guess)

def hv_at_fraction_of_plateau(eps0, eps_max, hv50, k, frac):
    frac = min(max(float(frac), 1e-6), 1 - 1e-6)
    return hv50 + k * np.log(frac / (1.0 - frac))

def load_summary_series(summary_dir, use_roi=True):
    paths = sorted(Path(summary_dir).glob("summary_HV*.json"))
    hv, eff, err = [], [], []
    for p in paths:
        d = json.loads(p.read_text())
        hv.append(float(d["hv_kv"]))
        if use_roi:
            eff.append(float(d["roi_efficiency"]["eff"]))
            err.append(float(d["roi_efficiency"]["err_binom"]))
        else:
            eff.append(float(d["global_efficiency"]["eff"]))
            err.append(float(d["global_efficiency"]["err_binom"]))
    order = np.argsort(hv)
    return np.asarray(hv, float)[order], np.asarray(eff, float)[order], np.asarray(err, float)[order]

def plot_cms_style(summary_dir, out_png, use_roi=True,
                   cms_left="CMS Preliminary", cms_right="RPC laboratory setup",
                   textbox="Pseudo-tracklet efficiency scan",
                   ymax=110.0, wp_frac=0.95, wp_offset_v=100.0):
    hv, eff, err = load_summary_series(summary_dir, use_roi=use_roi)
    eps0, eps_max, hv50, k = fit_sigmoid(hv, eff)
    plateau_pct = 100.0 * eps_max
    hv95 = hv_at_fraction_of_plateau(eps0, eps_max, hv50, k, wp_frac)
    wp_kv = hv95 + wp_offset_v / 1000.0
    wp_v = 1000.0 * wp_kv
    eff_wp = 100.0 * sigmoid(wp_kv, eps0, eps_max, hv50, k)

    fig = plt.figure(figsize=(7.2, 7.2))
    ax = plt.gca()
    ax.set_xlabel("HV [kV]", fontsize=18)
    ax.set_ylabel("Efficiency [%]", fontsize=22)
    ax.set_xlim(6.0, 7.5)
    ax.set_ylim(0.0, ymax)
    ax.grid(True, which="both", alpha=0.25)
    ax.minorticks_on()
    ax.axhline(100.0, linestyle="--", linewidth=1.2, alpha=0.6)
    ax.text(0.02, 1.03, cms_left, transform=ax.transAxes, fontsize=22, fontweight="bold", va="bottom")
    ax.text(0.98, 1.03, cms_right, transform=ax.transAxes, fontsize=18, ha="right", va="bottom")
    ax.text(0.10, 0.55, textbox, transform=ax.transAxes, fontsize=14, va="top", ha="left")
    ax.errorbar(hv, 100.0 * eff, yerr=100.0 * err, fmt="x", capsize=3,
                label=f"plateau = {plateau_pct:.2f}% | WP = {wp_v:.1f} V | Eff(WP) = {eff_wp:.2f}%")
    hv_fine = np.linspace(6.0, 7.5, 400)
    ax.plot(hv_fine, 100.0 * sigmoid(hv_fine, eps0, eps_max, hv50, k), linewidth=1.5)
    ax.legend(loc="upper left", fontsize=10, frameon=False)
    plt.tight_layout(); plt.savefig(out_png, dpi=200); plt.show()
    return {
        "eps0": eps0, "eps_max": eps_max, "hv50": hv50, "k": k,
        "plateau_percent": plateau_pct,
        "working_point_kv": wp_kv, "working_point_v": wp_v,
        "efficiency_at_wp_percent": eff_wp,
    }


In [ ]:

# Example:
# fit_info = plot_cms_style(
#     summary_dir=SUM_DIR,
#     out_png=PLOT_DIR / "efficiency_cms_style.png",
#     use_roi=True,
#     cms_right="CERN RPC test stand",
#     textbox="3-ch pseudo-tracklet scan
ROI: scintillator overlap",
# )
# fit_info


## 8. Optional: call the repository pipeline directly from the notebook

In [ ]:

REPO_SRC = Path("src")

def run_repo_pipeline(base_out, chamber_bases, z_positions,
                      anchor_index=0, target_index=-1,
                      regime="normal", hit_mode="earliest",
                      roi_name="scint_overlap", roi_rect=(0, 22, 0, 130),
                      dynamic_cuts=False):
    cmd = [
        "python3", str(REPO_SRC / "rpc_hv_pipeline.py"),
        "--python", "python3",
        "--producer", str(REPO_SRC / "rpc_tracklet_efficiency.py"),
        "--summarizer", str(REPO_SRC / "rpc_make_eff_summary.py"),
        "--plotter", str(REPO_SRC / "rpc_efficiency_vs_hv.py"),
        "--base-out", str(base_out),
        "--regime", regime,
        "--hit-mode", hit_mode,
        "--mode", "optionc",
        "--anchor-index", str(anchor_index),
        "--target-index", str(target_index),
        "--roi-name", roi_name,
        "--roi-rect", *[str(v) for v in roi_rect],
        "--z", *[str(v) for v in z_positions],
        "--chambers", *[str(v) for v in chamber_bases],
    ]
    if dynamic_cuts:
        cmd += ["--dynamic-cuts", "--cut-estimator", str(REPO_SRC / "rpc_estimate_tracklet_cuts.py")]
    print(" ".join(cmd))
    return subprocess.run(cmd, check=False, capture_output=True, text=True)


In [ ]:

# Example:
# proc = run_repo_pipeline(
#     base_out="out_repo_pipeline",
#     chamber_bases=CHAMBER_BASES,
#     z_positions=Z_POSITIONS,
#     anchor_index=ANCHOR_INDEX,
#     target_index=TARGET_INDEX,
#     regime="normal",
#     hit_mode=HIT_MODE,
#     roi_name=ROI_NAME,
#     roi_rect=ROI_RECT,
#     dynamic_cuts=True,
# )
# print(proc.stdout)
# print(proc.stderr)


## 9. Optional: run the shell wrapper from the notebook

In [ ]:

# Example shell command:
# !bash run_rpc_hv_scan_cms.sh #   --python python3 #   --base-out out_hv_3ch #   --regime normal #   --hit-mode earliest #   --roi-name scint_overlap #   --roi-rect 0 22 0 130 #   --z 0 55 110 #   --anchor-index 0 #   --target-index 2 #   --chambers /path/to/ChamberA /path/to/ChamberB /path/to/ChamberC



## 10. Notes

- The notebook uses **anchor coordinates** to fill the denominator and numerator maps, matching the repository producer.
- The **target chamber never contributes to the denominator**.
- For `N >= 3`, the target position is predicted with a straight-line fit in `x(z)` and `y(z)`.
- The companion shell wrapper can execute the repository pipeline end-to-end and then generate a **CMS-style HV plot** from the JSON summaries.
